# 44 · Physiological state, tissue state, more human donors, and rodent protein (revision checks)

Answers four points of the internal referee report (v01):

| Section | Referee point | Question |
|---|---|---|
| A | D8, M4.5 | Are the mouse-only zonation programs (sterol, PPARα/fatty-acid oxidation, glutathione, tyrosine catabolism) a feeding or time-of-day state rather than a species trait? |
| B | M4.3, D7 | Where does our human donor sit on the injury axis of the Lake et al. 2023 donors? |
| C | D9 | Do human-side gradients replicate in additional human donors with spatial data (Abedini et al. 2024 Visium)? |
| D | D10 | What does rodent segment proteomics (rat, Kidney Tubule Expression Atlas) show for the position-defining genes? |

The protocol was written before this notebook ran (workstream 3, `revision.md`) and is
summarized in each section. All inputs are public except section B, which uses our own PT
structures. Everything is descriptive.

In [ ]:
import argparse
import json
import os
import sys
from pathlib import Path

start = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
project = next(p for p in (start, *start.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
external = data_root / 'external'
state_dir = external / 'kidney_physiological_state'
upstream13 = results_root / 'minimal_pt_scfates'
upstream38 = results_root / 'pt_literature_deep'
output = results_root / 'pt_revision_state'
figures, tables = output / 'figures', output / 'tables'
for folder in (figures, tables):
    folder.mkdir(parents=True, exist_ok=True)
LAKE = external / 'kpmp_lake2023' / 'lake2023_snRNA_v1.5.h5ad'
SN_SAMPLES = {'ZT4_1': 'GSM9744085_CTRL_ZT4_01', 'ZT4_2': 'GSM9744086_CTRL_ZT4_02',
              'ZT16_1': 'GSM9744089_CTRL_ZT16_01', 'ZT16_2': 'GSM9744090_CTRL_ZT16_02'}
required = [LAKE, upstream13 / 'cross_species_pt_scfates.h5ad', upstream13 / 'ortholog_map_used.csv',
            upstream38 / 'gene_level_lake_contrasts.csv', upstream38 / 'probe_counts.csv',
            results_root / 'mouse_only_v5' / 'all_mouse_tubules_harmony.h5ad',
            state_dir / 'GSE267280_RNA_Seq_counts.txt.gz', state_dir / 'GSE277302_Normalized_counts.csv.gz',
            state_dir / 'GSE277302_series_matrix.txt.gz',
            external / 'census_pt_segments' / 'mouse_pt_segment_pseudobulk_counts.csv.gz',
            external / 'abedini2024_visium' / 'st_counts.mtx', external / 'abedini2024_visium' / 'GSE211785_ST_metadata.txt.gz',
            external / 'ktea_rat_proteomics' / 'KTEA_proteome_individual_sample.txt',
            *[state_dir / f'{p}_{f}' for p in SN_SAMPLES.values() for f in ('barcodes.tsv.gz', 'features.tsv.gz', 'matrix.mtx.gz')]]
for path in required:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path}')
NOTEBOOK_LOGIC_VERSION = '44.revision_state.1'
PROGRAMS = {
    'sterol / SREBP2': ['Hmgcr', 'Hmgcs1', 'Sqle', 'Cyp51', 'Idi1', 'Fdft1', 'Mvd', 'Ldlr', 'Insig1', 'Dhcr7', 'Lss', 'Msmo1'],
    'PPARα / fatty-acid oxidation': ['Acox1', 'Ehhadh', 'Cyp4a14', 'Cyp4a10', 'Acot1', 'Crot', 'Nudt19', 'Pdk4', 'Hmgcs2',
                                     'Cpt1a', 'Acadm', 'Slc27a2', 'Acsm3'],
    'glutathione synthesis': ['Gclc', 'Gclm', 'Gss', 'Gsr'],
    'tyrosine catabolism': ['Tat', 'Hpd', 'Hgd', 'Fah', 'Gstz1', 'Pah'],
    'lead genes': ['Gatm', 'Gamt', 'Acaa2', 'Dcxr', 'Ugt3a1', 'Acox2'],
    'structural markers': ['Slc5a2', 'Slc5a12', 'Slc7a13', 'Slc22a7', 'Slc34a1', 'Lrp2'],
}
TESTED = {g for genes in PROGRAMS.values() for g in genes}
print('Results folder:', output)

## A1 · Whole-kidney effects of fasting and time of day, against the species difference

Fasting: GSE267280 (one pooled column per condition, so a point estimate). Time of day:
GSE277302, active (ZT18) versus inactive (ZT6) period, six mice per sex per time. The comparator
is each gene's external species × position interaction, |Lake human S3 − S1 − male mouse snRNA
S3 − S1| (notebook 38). Decision per program (fixed): the state effect is *comparable* if the
program median of the larger state effect is at least half the program median |interaction|.

In [ ]:
import gzip
import numpy as np
import pandas as pd
from scipy import io, sparse, stats
from IPython.display import display

features = pd.read_csv(state_dir / f'{SN_SAMPLES["ZT4_1"]}_features.tsv.gz', sep='\t', header=None, names=['id', 'symbol', 'type'])
ens_to_symbol = dict(zip(features.id, features.symbol))
fast = pd.read_csv(state_dir / 'GSE267280_RNA_Seq_counts.txt.gz', sep='\t')
fast['symbol'] = fast.Gene_id.str.split('.').str[0].map(ens_to_symbol)
fast = fast.dropna(subset=['symbol']).groupby('symbol')[['Fed_Kidney', 'Fasting_Kidney']].sum()
fast_cpm = fast / fast.sum() * 1e6
fasting_lfc = np.log2(fast_cpm.Fasting_Kidney + 1) - np.log2(fast_cpm.Fed_Kidney + 1)

with gzip.open(state_dir / 'GSE277302_series_matrix.txt.gz', 'rt') as handle:
    lines = [l.rstrip('\n').split('\t') for l in handle if l.startswith('!Sample_title') or l.startswith('!Sample_characteristics_ch1')]
titles = [x.strip('"') for x in lines[0][1:]]
characteristics = {row[1].strip('"').split(':')[0]: [x.strip('"').split(': ')[1] for x in row[1:]] for row in lines[1:]}
diurnal_meta = pd.DataFrame({'sex': characteristics['Sex'], 'period': characteristics['treatment']}, index=titles)
diurnal = pd.read_csv(state_dir / 'GSE277302_Normalized_counts.csv.gz', index_col=0)
diurnal['symbol'] = diurnal.EnsemblID.map(ens_to_symbol)
diurnal = diurnal.dropna(subset=['symbol']).groupby('symbol')[titles].sum()
diurnal_lfc, diurnal_p = {}, {}
for sex in ('male', 'female'):
    active = diurnal_meta.index[diurnal_meta.sex.eq(sex) & diurnal_meta.period.eq('active')]
    inactive = diurnal_meta.index[diurnal_meta.sex.eq(sex) & diurnal_meta.period.eq('inactive')]
    log = np.log2(diurnal + 1)
    diurnal_lfc[sex] = log[active].mean(axis=1) - log[inactive].mean(axis=1)
    diurnal_p[sex] = pd.Series(stats.ttest_ind(log[active], log[inactive], axis=1).pvalue, index=diurnal.index)

contrasts = pd.read_csv(upstream38 / 'gene_level_lake_contrasts.csv', index_col=0)
interaction = (contrasts.lake_human_S3_minus_S1 - contrasts.mouse_sn_male_S3_minus_S1)
rows = []
for program, genes in PROGRAMS.items():
    for g in genes:
        rows.append({'program': program, 'gene': g, 'fasting_log2fc': fasting_lfc.get(g, np.nan),
                     'diurnal_log2fc_male': diurnal_lfc['male'].get(g, np.nan), 'diurnal_p_male': diurnal_p['male'].get(g, np.nan),
                     'diurnal_log2fc_female': diurnal_lfc['female'].get(g, np.nan), 'diurnal_p_female': diurnal_p['female'].get(g, np.nan),
                     'mouse_sn_male_S3_minus_S1': contrasts.mouse_sn_male_S3_minus_S1.get(g, np.nan),
                     'lake_human_S3_minus_S1': contrasts.lake_human_S3_minus_S1.get(g, np.nan),
                     'species_interaction': interaction.get(g, np.nan)})
state_genes = pd.DataFrame(rows)
state_genes['largest_state_effect'] = state_genes[['fasting_log2fc', 'diurnal_log2fc_male', 'diurnal_log2fc_female']].abs().max(axis=1)
state_programs = state_genes.groupby('program', sort=False).agg(
    genes=('gene', 'size'), median_largest_state_effect=('largest_state_effect', 'median'),
    median_abs_interaction=('species_interaction', lambda s: s.abs().median()),
    median_abs_mouse_zonation=('mouse_sn_male_S3_minus_S1', lambda s: s.abs().median()))
state_programs['state ÷ interaction'] = state_programs.median_largest_state_effect / state_programs.median_abs_interaction
state_programs['A1 decision'] = np.where(state_programs['state ÷ interaction'] >= .5, 'state effect comparable',
                                         'small relative to the species difference')
state_genes.to_csv(tables / 'A1_state_effects_by_gene.csv', index=False)
state_programs.to_csv(tables / 'A1_state_effects_by_program.csv')
display(state_programs.round(2))

## A2 · Does PT zonation of these programs hold at two times of day?

GSE331332 control mice at ZT4 (rest) and ZT16 (active, feeding), two each, snRNA without cell
labels. Segment markers come from the Census mouse atlas (a different lab) and exclude every
tested gene. PT nuclei pass the Otsu threshold of a canonical PT score; S1 and S3 nuclei are the
two extremes of the within-sample S1-versus-S3 marker score (margin 0.5 SD).

In [ ]:
census_counts = pd.read_csv(external / 'census_pt_segments' / 'mouse_pt_segment_pseudobulk_counts.csv.gz', index_col=0)
census_meta = pd.read_csv(external / 'census_pt_segments' / 'mouse_pt_segment_pseudobulk_meta.csv', index_col=0)
census_meta = census_meta[~census_meta.development_stage.str.contains('embryo|E16|Theiler', case=False)]
census_meta['segment'] = census_meta.cell_type.map({'epithelial cell of proximal tubule segment 1': 'S1',
                                                    'epithelial cell of proximal tubule segment 2': 'S2',
                                                    'epithelial cell of proximal tubule segment 3': 'S3'})
census_meta = census_meta[census_meta.segment.notna() & census_meta.n_cells.ge(50)]
male = census_meta[census_meta.sex.eq('male')]
seg_cpm = {s: (census_counts.loc[male.index[male.segment.eq(s)]].sum() / male.loc[male.segment.eq(s), 'total_counts_all_genes'].sum() * 1e6)
           for s in ('S1', 'S3')}
diff = np.log2(seg_cpm['S1'] + 1) - np.log2(seg_cpm['S3'] + 1)
eligible = ~diff.index.isin(TESTED)
S1_MARKERS = diff[eligible & seg_cpm['S1'].ge(100)].nlargest(10).index.tolist()
S3_MARKERS = (-diff)[eligible & seg_cpm['S3'].ge(100)].nlargest(10).index.tolist()
PT_MARKERS = ['Lrp2', 'Cubn', 'Slc34a1', 'Hnf4a']
print('S1 markers:', S1_MARKERS)
print('S3 markers:', S3_MARKERS)


def otsu(values, bins=256):
    hist, edges = np.histogram(values, bins=bins)
    centres = (edges[:-1] + edges[1:]) / 2
    w1, w2 = np.cumsum(hist), np.cumsum(hist[::-1])[::-1]
    m1 = np.cumsum(hist * centres) / np.maximum(w1, 1)
    m2 = (np.cumsum((hist * centres)[::-1]) / np.maximum(np.cumsum(hist[::-1]), 1))[::-1]
    between = w1[:-1] * w2[1:] * (m1[:-1] - m2[1:]) ** 2
    return centres[:-1][np.argmax(between)]


def sn_sample(prefix):
    matrix = sparse.csc_matrix(io.mmread(state_dir / f'{prefix}_matrix.mtx.gz'))     # genes x nuclei
    feats = pd.read_csv(state_dir / f'{prefix}_features.tsv.gz', sep='\t', header=None, names=['id', 'symbol', 'type'])
    totals = np.asarray(matrix.sum(axis=0)).ravel()
    keep = totals >= 500
    matrix, totals = matrix[:, keep], totals[keep]
    symbols = feats.symbol.to_numpy()

    def score(genes):
        idx = [i for i, s in enumerate(symbols) if s in set(genes)]
        block = matrix[idx].toarray() / totals * 1e4
        return np.log1p(block).mean(axis=0)
    pt_score = score(PT_MARKERS)
    pt = pt_score > otsu(pt_score)
    z1, z3 = score(S1_MARKERS)[pt], score(S3_MARKERS)[pt]
    d = (z1 - z1.mean()) / z1.std() - (z3 - z3.mean()) / z3.std()
    labels = np.full(pt.sum(), '', dtype=object)
    labels[d > .5], labels[d < -.5] = 'S1', 'S3'
    sub = matrix[:, np.flatnonzero(pt)]
    out = {}
    for seg in ('S1', 'S3'):
        cols = np.flatnonzero(labels == seg)
        counts = np.asarray(sub[:, cols].sum(axis=1)).ravel()
        out[seg] = pd.Series(counts, index=symbols).groupby(level=0).sum() / totals[pt][cols].sum() * 1e6
    return out, {'nuclei': int(keep.sum()), 'pt_nuclei': int(pt.sum()), 'S1': int((labels == 'S1').sum()), 'S3': int((labels == 'S3').sum())}


sn_contrast, sn_counts = {}, {}
for label, prefix in SN_SAMPLES.items():
    cpm, n = sn_sample(prefix)
    sn_contrast[label] = np.log2(cpm['S3'] + 1) - np.log2(cpm['S1'] + 1)
    sn_counts[label] = n
sn_contrast = pd.DataFrame(sn_contrast)
display(pd.DataFrame(sn_counts))

a2_rows = []
for program, genes in PROGRAMS.items():
    block = sn_contrast.reindex(genes).dropna(how='all')
    zt4, zt16 = block[['ZT4_1', 'ZT4_2']].mean(axis=1), block[['ZT16_1', 'ZT16_2']].mean(axis=1)
    zoned = (zt4.abs() >= .5) | (zt16.abs() >= .5)
    same_sign = (np.sign(block[zoned]).nunique(axis=1) == 1)
    ratio = zt16.abs()[zoned].median() / zt4.abs()[zoned].median() if zoned.any() else np.nan
    change = (zt16 - zt4).abs().median()
    inter = interaction.reindex(genes).abs().median()
    a2_rows.append({'program': program, 'genes measured': len(block), 'zonated genes': int(zoned.sum()),
                    'share with sign stable in all 4 samples': float(same_sign.mean()) if zoned.any() else np.nan,
                    'median |S3−S1| ZT4': zt4.abs()[zoned].median(), 'median |S3−S1| ZT16': zt16.abs()[zoned].median(),
                    'amplitude ratio ZT16 ÷ ZT4': ratio, 'median |ZT16 − ZT4| change': change,
                    'median |species interaction|': inter})
a2 = pd.DataFrame(a2_rows).set_index('program')
a2['zonation state-stable'] = (a2['share with sign stable in all 4 samples'] >= .8) & a2['amplitude ratio ZT16 ÷ ZT4'].between(.67, 1.5)
a2['time of day could explain ≥ half the species gap'] = a2['median |ZT16 − ZT4| change'] >= .5 * a2['median |species interaction|']
a2_genes = sn_contrast.reindex(sorted(TESTED)).assign(program=lambda d: [next(p for p, g in PROGRAMS.items() if i in g) for i in d.index])
a2_genes.to_csv(tables / 'A2_snrna_time_of_day_S3_minus_S1_by_gene.csv')
a2.to_csv(tables / 'A2_snrna_time_of_day_by_program.csv')
display(a2.round(2))

## B · Our donor on the Lake injury axis

Signature: genes up or down in Lake aPT nuclei relative to healthy PT nuclei of the same donor.
A sample's score is the mean within-sample percentile of up genes minus that of down genes, which
does not depend on library size. The platform offset between our Visium HD data and snRNA is
estimated in mouse (our control mice against healthy Census mouse donors) and assumed to transfer
to human.

In [ ]:
import h5py
import anndata as ad
from pseudospace.replication_rules import csr_group_sums
from pseudospace.stage_cache import cached_payload, digest
from pseudospace.pathway_inputs import rebuild_pt_expression

orthologs = pd.read_csv(upstream13 / 'ortholog_map_used.csv')
human_var = ad.read_h5ad(data_root / 'tubule_by_gene' / 'HUK1_COR1_tubule_by_gene_caleb.h5ad', backed='r')
human_ids = human_var.var.gene_ids.copy()
human_var.file.close()


def obs_column(group, name):
    item = group[name]
    if isinstance(item, h5py.Group):
        categories = np.array([c.decode() if isinstance(c, bytes) else str(c) for c in item['categories'][:]])
        return pd.Categorical.from_codes(item['codes'][:], categories).astype(str)
    return item[:].astype(str)


def lake_states():
    with h5py.File(LAKE, 'r') as handle:
        obs = pd.DataFrame({c: obs_column(handle['obs'], c) for c in ['subclass.l2', 'disease', 'region', 'tissue', 'donor_id']})
        keep = (obs.region.eq('C') & obs.tissue.eq('cortex of kidney')
                & obs['subclass.l2'].isin(['PT-S1', 'PT-S2', 'PT-S3', 'aPT', 'dPT', 'cycPT']))
        label = (obs.donor_id + '|' + obs.disease + '|' + obs['subclass.l2'])[keep]
        groups, codes = np.unique(label.to_numpy(), return_inverse=True)
        raw = handle['raw/X']
        var_ids = handle['raw/var/_index'][:].astype(str)
        var_names = np.asarray(obs_column(handle['raw/var'], 'feature_name'))
        sums, totals, counts = csr_group_sums(raw['data'], raw['indices'], raw['indptr'], np.flatnonzero(keep.to_numpy()),
                                              codes, len(groups), len(var_ids), block_rows=5000)
    by_id = dict(zip(orthologs.human_symbol.map(human_ids), orthologs.mouse_symbol))
    by_symbol = dict(zip(orthologs.human_symbol, orthologs.mouse_symbol))
    target = np.array([by_id.get(i, by_symbol.get(n, '')) for i, n in zip(var_ids, var_names)])
    mapped = target != ''
    frame = pd.DataFrame(sums[:, mapped], columns=target[mapped]).T.groupby(level=0).sum().T
    return {'groups': groups.astype(str), 'counts': counts, 'totals': totals, 'sums': frame.to_numpy(),
            'genes': np.asarray(frame.columns, dtype=str)}


lake = cached_payload('lake_cortex_pt_states', lake_states, root=output / 'stage_cache',
                      params={'labels': 'PT-S1,PT-S2,PT-S3,aPT,dPT,cycPT', 'region': 'C cortex'},
                      inputs={'lake': [LAKE.name, LAKE.stat().st_size], 'orthologs': digest(orthologs)},
                      code=digest([NOTEBOOK_LOGIC_VERSION]))
lake_meta = pd.DataFrame([g.split('|') for g in lake['groups']], columns=['donor', 'disease', 'state'])
lake_meta['nuclei'], lake_meta['total'] = lake['counts'], lake['totals']
lake_sums = pd.DataFrame(lake['sums'], columns=lake['genes'])

# Our PT structures (human sections and control mice) and the four mouse-only specimens (with AKI).
saved_pt = ad.read_h5ad(upstream13 / 'cross_species_pt_scfates.h5ad', backed='r')
pt_obs = saved_pt.obs.copy()
saved_pt.file.close()
pt_obs = pt_obs[pt_obs.broad_tubule_marker_call.astype(str).eq('PT')].copy()
expected = {'mouse': ['Ctrl1A2', 'Ctrl1A4'], 'human': ['HUK1_COR1', 'HUK1_MED1']}
ours_ad = rebuild_pt_expression(pt_obs, {n: data_root / 'tubule_by_gene' / f'{n}_tubule_by_gene_caleb.h5ad'
                                        for v in expected.values() for n in v}, orthologs, target_sum=1e4)
ours_counts = sparse.csr_matrix(ours_ad.layers['counts'])
ours_specimen = ours_ad.obs['sample'].astype(str).to_numpy()
ours_pb = pd.DataFrame({n: np.asarray(ours_counts[ours_specimen == n].sum(axis=0)).ravel() for n in sum(expected.values(), [])},
                       index=ours_ad.var_names)
mouse_only = ad.read_h5ad(results_root / 'mouse_only_v5' / 'all_mouse_tubules_harmony.h5ad', backed='r')
mo_obs = mouse_only.obs[['sample', 'segment_class']].copy()
mo_keep = np.flatnonzero(mo_obs.segment_class.astype(str).isin(['PT-S1', 'PT-S2', 'PT-S3']).to_numpy())
mo_counts = sparse.csr_matrix(mouse_only.layers['counts'][mo_keep])
mo_genes = mouse_only.var_names.copy()
mouse_only.file.close()
mo_sample = mo_obs['sample'].astype(str).to_numpy()[mo_keep]
mo_pb = pd.DataFrame({n: np.asarray(mo_counts[mo_sample == n].sum(axis=0)).ravel() for n in ['Ctrl1A2', 'Ctrl1A4', 'IR2A2', 'IR2A4']},
                     index=mo_genes)
census_pb = {}
for donor, block in census_meta.groupby('donor_id'):
    census_pb[donor] = census_counts.loc[block.index].sum()
census_pb = pd.DataFrame(census_pb)

universe = (pd.Index(ours_pb.index).intersection(mo_pb.index).intersection(lake_sums.columns).intersection(census_pb.index))
print(f'Gene universe U: {len(universe):,} genes')


def lake_state_pb(donor, states):
    rows = lake_meta.index[lake_meta.donor.eq(donor) & lake_meta.state.isin(states)]
    return lake_sums.loc[rows].sum(), lake_meta.loc[rows, 'total'].sum(), lake_meta.loc[rows, 'nuclei'].sum()


healthy_states = ['PT-S1', 'PT-S2', 'PT-S3']
signature_rows = {}
for donor in lake_meta.donor.unique():
    a_sum, a_tot, a_n = lake_state_pb(donor, ['aPT'])
    h_sum, h_tot, h_n = lake_state_pb(donor, healthy_states)
    if a_n >= 30 and h_n >= 100:
        signature_rows[donor] = {'a_cpm': a_sum[universe] / a_tot * 1e6, 'h_cpm': h_sum[universe] / h_tot * 1e6}
a_cpm = pd.DataFrame({d: v['a_cpm'] for d, v in signature_rows.items()})
h_cpm = pd.DataFrame({d: v['h_cpm'] for d, v in signature_rows.items()})
lfc = (np.log2(a_cpm + 1) - np.log2(h_cpm + 1)).median(axis=1)
UP = lfc[a_cpm.mean(axis=1).ge(20)].nlargest(30).index.tolist()
DOWN = lfc[h_cpm.mean(axis=1).ge(20)].nsmallest(30).index.tolist()
print(f'Signature from {len(signature_rows)} Lake donors. Up: {UP}')
print(f'Down: {DOWN}')


def injury_score(pseudobulk):
    ranks = pseudobulk.reindex(universe).fillna(0).rank(pct=True)
    return float(ranks[UP].mean() - ranks[DOWN].mean())


lake_rows = []
for donor in lake_meta.donor.unique():
    total_sum, total, n = lake_state_pb(donor, healthy_states + ['aPT', 'dPT', 'cycPT'])
    altered = lake_meta.loc[lake_meta.donor.eq(donor) & lake_meta.state.isin(['aPT', 'dPT', 'cycPT']), 'nuclei'].sum()
    if n >= 100:
        lake_rows.append({'donor': donor, 'disease': lake_meta.loc[lake_meta.donor.eq(donor), 'disease'].iloc[0],
                          'pt_nuclei': int(n), 'altered_fraction': altered / n, 'injury_score': injury_score(total_sum)})
lake_scores = pd.DataFrame(lake_rows)
rho = stats.spearmanr(lake_scores.injury_score, lake_scores.altered_fraction).statistic
census_scores = pd.Series({d: injury_score(census_pb[d]) for d in census_pb.columns})
ours_scores = pd.Series({n: injury_score(ours_pb[n]) for n in ours_pb.columns})
mo_scores = pd.Series({n: injury_score(mo_pb[n]) for n in mo_pb.columns})
offset = float(ours_scores[expected['mouse']].mean() - census_scores.median())
normal = lake_scores[lake_scores.disease.eq('normal')]
fit = np.polyfit(lake_scores.injury_score, lake_scores.altered_fraction, 1)
placement = pd.DataFrame({
    'raw score': ours_scores[expected['human']],
    'calibrated score': ours_scores[expected['human']] - offset})
placement['percentile among Lake normal donors'] = [float((normal.injury_score < s).mean()) for s in placement['calibrated score']]
placement['predicted altered fraction (Lake fit)'] = np.polyval(fit, placement['calibrated score'])
placement['within Lake normal range'] = placement['calibrated score'].between(normal.injury_score.min(), normal.injury_score.max())
placement['above Lake AKI median'] = placement['calibrated score'] > lake_scores.loc[lake_scores.disease.eq('acute kidney injury'), 'injury_score'].median()
b_summary = pd.Series({
    'Lake donors scored': len(lake_scores), 'Spearman score vs altered fraction (Lake)': rho,
    'Lake normal median': normal.injury_score.median(), 'Lake normal range': f'{normal.injury_score.min():.3f} to {normal.injury_score.max():.3f}',
    'Lake CKD median': lake_scores.loc[lake_scores.disease.eq('chronic kidney disease'), 'injury_score'].median(),
    'Lake AKI median': lake_scores.loc[lake_scores.disease.eq('acute kidney injury'), 'injury_score'].median(),
    'Census healthy mouse median': census_scores.median(), 'our control mice (mean)': ours_scores[expected['mouse']].mean(),
    'platform offset (mouse)': offset, 'mouse-only: AKI − control': mo_scores[['IR2A2', 'IR2A4']].mean() - mo_scores[['Ctrl1A2', 'Ctrl1A4']].mean()})
lake_scores.to_csv(tables / 'B_lake_donor_injury_scores.csv', index=False)
placement.to_csv(tables / 'B_our_donor_placement.csv')
pd.DataFrame({'gene': UP + DOWN, 'direction': ['up in aPT'] * len(UP) + ['down in aPT'] * len(DOWN),
              'median_log2fc_aPT_vs_healthy': lfc[UP + DOWN].to_numpy()}).to_csv(tables / 'B_injury_signature.csv', index=False)
pd.concat({'census mouse (healthy)': census_scores, 'mouse-only Visium HD': mo_scores, 'ours (rebuild)': ours_scores}).to_csv(tables / 'B_calibration_scores.csv')
display(b_summary.to_frame('value'))
display(placement.round(3))

## C · More human donors: Abedini et al. 2024 Visium (GSE211785)

Samples with at least 50 spots labelled PT_S1 and 50 labelled PT_S3 by the authors. Spots are
55 µm and mix cells, so contrasts are diluted; signs, not amplitudes, are the main readout.

In [ ]:
st_dir = external / 'abedini2024_visium'
st_meta = pd.read_csv(st_dir / 'GSE211785_ST_metadata.txt.gz', sep='\t', index_col=0)


def load_visium():
    matrix = sparse.csc_matrix(io.mmread(st_dir / 'st_counts.mtx'))
    genes = (st_dir / 'st_genes.txt').read_text().split('\n')[:matrix.shape[0]]
    barcodes = (st_dir / 'st_barcodes.txt').read_text().split('\n')[:matrix.shape[1]]
    meta = st_meta.reindex(barcodes)
    totals = np.asarray(matrix.sum(axis=0)).ravel()
    groups = (meta['orig.ident'].astype(str) + '|' + meta.celltype.astype(str)).to_numpy()
    keep = np.isin(meta.celltype.astype(str), ['PT_S1', 'PT_S3'])
    labels, codes = np.unique(groups[keep], return_inverse=True)
    design = sparse.csr_matrix((np.ones(keep.sum()), (codes, np.arange(keep.sum()))), shape=(len(labels), keep.sum()))
    sums = (design @ matrix[:, np.flatnonzero(keep)].T).toarray()
    return {'labels': labels.astype(str), 'sums': sums, 'totals': np.asarray(design @ totals[keep]).ravel(),
            'spots': np.asarray(design.sum(axis=1)).ravel(), 'genes': np.asarray(genes, dtype=str)}


st = cached_payload('abedini_pt_pseudobulk', load_visium, root=output / 'stage_cache',
                    inputs={'mtx': [(st_dir / 'st_counts.mtx').stat().st_size]}, code=digest([NOTEBOOK_LOGIC_VERSION]))
to_mouse = dict(zip(orthologs.human_symbol, orthologs.mouse_symbol))
st_frame = pd.DataFrame(st['sums'], columns=st['genes'])
st_frame.columns = [to_mouse.get(g, '') for g in st_frame.columns]
st_frame = st_frame.loc[:, st_frame.columns != ''].T.groupby(level=0).sum().T
st_info = pd.DataFrame([l.split('|') for l in st['labels']], columns=['sample', 'segment'])
st_info['spots'], st_info['total'] = st['spots'], st['totals']
status = st_meta.groupby('orig.ident').Status.first()
st_samples = [s for s in st_info['sample'].unique()
              if st_info.loc[st_info['sample'].eq(s), 'spots'].min() >= 50 and st_info.loc[st_info['sample'].eq(s)].shape[0] == 2]
st_contrast = {}
for s in st_samples:
    i1 = st_info.index[(st_info['sample'] == s) & (st_info.segment == 'PT_S1')][0]
    i3 = st_info.index[(st_info['sample'] == s) & (st_info.segment == 'PT_S3')][0]
    st_contrast[s] = (np.log2(st_frame.loc[i3] / st_info.total[i3] * 1e6 + 1) - np.log2(st_frame.loc[i1] / st_info.total[i1] * 1e6 + 1))
st_contrast = pd.DataFrame(st_contrast)
print(f'{len(st_samples)} samples ({status[st_samples].value_counts().to_dict()})')

positive = {'Slc5a2': -1, 'Slc5a1': 1, 'Slc7a13': 1, 'Slc22a7': 1}
label_check = pd.DataFrame({g: (np.sign(st_contrast.loc[g]) == s) for g, s in positive.items() if g in st_contrast.index}).T
print('Label check (share of samples with the expected sign):', label_check.mean(axis=1).round(2).to_dict())

LEADS_C = ['Gatm', 'Gamt', 'Acadm', 'Acaa2', 'Dcxr', 'Ugt3a1', 'Acox2', 'Gclc', 'Gclm', 'Gss', 'Ugt1a9', 'Rbp4', 'Aox1',
           'Slc6a19', 'Slc22a6', 'Slc13a3']
mouse_mean = (contrasts.mouse_sn_male_S3_minus_S1 + contrasts.mouse_sn_female_S3_minus_S1) / 2
G = contrasts.index[contrasts.mouse_sn_male_S3_minus_S1.abs().ge(1) & contrasts.lake_max_cpm.ge(10)]
G = G.intersection(st_contrast.index)


def slope(y, x):
    ok = np.isfinite(y) & np.isfinite(x)
    return float((y[ok] * x[ok]).sum() / (x[ok] ** 2).sum())


k_samples = pd.Series({s: slope(st_contrast.loc[G, s].to_numpy(), contrasts.mouse_sn_male_S3_minus_S1.loc[G].to_numpy()) for s in st_samples})
k_bar = float(k_samples.median())
c_rows = []
for g in LEADS_C:
    if g not in st_contrast.index:
        c_rows.append({'gene': g, 'note': 'not in the Visium matrix'})
        continue
    values = st_contrast.loc[g, st_samples]
    lake_h = contrasts.lake_human_S3_minus_S1.get(g, np.nan)
    informative = values[values.abs() >= .1]
    agree = float((np.sign(informative) == np.sign(lake_h)).mean()) if len(informative) else np.nan
    obs_, se = float(values.mean()), float(values.std(ddof=1) / np.sqrt(len(values)))
    m = float(mouse_mean.get(g, np.nan))
    pred = k_bar * m
    same = int((np.sign(values) == np.sign(obs_)).sum())
    n = len(values)
    if abs(m) >= .25 and np.sign(obs_) != np.sign(m) and abs(obs_) >= .25 and same >= .8 * n:
        klass = 'reversal beyond flattening'
    elif abs(m) < .25 and abs(obs_) >= .5 and same >= .8 * n:
        klass = 'human-specific gradient'
    elif np.sign(obs_) == np.sign(m) and abs(obs_) > abs(pred) and abs(obs_ - pred) > 2 * se:
        klass = 'exceeds flattening'
    elif abs(obs_) < abs(pred) and abs(obs_ - pred) > 2 * se:
        klass = 'flatter than predicted'
    else:
        klass = 'consistent with flattening'
    c_rows.append({'gene': g, 'lake_human_S3_minus_S1': lake_h, 'visium_mean': obs_, 'visium_se': se,
                   'samples_same_sign_as_mean': f'{same}/{n}', 'informative_samples': len(informative),
                   'share_agreeing_with_lake_sign': agree,
                   'human_arm_replicates': (len(informative) >= 5 and agree >= .7) if abs(lake_h) >= .25 else None,
                   'mouse_sn_mean': m, 'predicted_flattening': pred, 'class_visium': klass,
                   'control_samples_mean': float(values[[s for s in st_samples if status[s] == 'Control']].mean())})
c_table = pd.DataFrame(c_rows)
st_contrast.loc[[g for g in LEADS_C + list(positive) if g in st_contrast.index]].to_csv(tables / 'C_visium_S3_minus_S1_by_sample.csv')
c_table.to_csv(tables / 'C_visium_lead_genes.csv', index=False)
k_samples.rename('k').to_frame().assign(status=status.reindex(k_samples.index)).to_csv(tables / 'C_visium_amplitude_k.csv')
print(f'Amplitude k on mouse-zonated genes (|male snRNA S3 − S1| ≥ 1, n = {len(G)}): median {k_bar:.3f}, '
      f'range {k_samples.min():.3f}–{k_samples.max():.3f}')
display(c_table.round(2))

### C, post hoc (added after the pre-specified label check failed)

The pre-specified label check failed: in PT_S3 spots almost every PT gene is lower than in PT_S1
spots, including the S3 markers. That pattern is what dilution looks like. Late-PT spots contain
less PT and more of neighbouring cells, so the whole PT transcriptome drops. As a post-hoc,
exploratory correction, each sample's contrasts are centred on the median S3 − S1 of
PT-expressed genes that are zonated in neither species (Lake CPM ≥ 100, |Lake S3 − S1| < 0.25
and |male mouse snRNA S3 − S1| < 0.25). The label check and the lead genes are then repeated.
Multiplicative dilution is not corrected, so amplitudes stay compressed.

In [ ]:
flat_genes = contrasts.index[contrasts.lake_max_cpm.ge(100) & contrasts.lake_human_S3_minus_S1.abs().lt(.25)
                             & contrasts.mouse_sn_male_S3_minus_S1.abs().lt(.25)].intersection(st_contrast.index)
offsets = st_contrast.loc[flat_genes].median()
centred = st_contrast - offsets
label_check_c = pd.DataFrame({g: (np.sign(centred.loc[g]) == s) for g, s in positive.items() if g in centred.index}).T
print(f'{len(flat_genes)} reference genes; per-sample offsets (log2):', offsets.round(2).to_dict())
print('Label check after centring:', label_check_c.mean(axis=1).round(2).to_dict())
post_rows = []
for g in LEADS_C:
    if g not in centred.index:
        continue
    values = centred.loc[g, st_samples]
    lake_h = contrasts.lake_human_S3_minus_S1.get(g, np.nan)
    informative = values[values.abs() >= .1]
    post_rows.append({'gene': g, 'lake_human_S3_minus_S1': lake_h, 'mouse_sn_mean': float(mouse_mean.get(g, np.nan)),
                      'centred_visium_mean': float(values.mean()), 'centred_visium_median': float(values.median()),
                      'informative_samples': len(informative),
                      'share_agreeing_with_lake_sign': float((np.sign(informative) == np.sign(lake_h)).mean()) if len(informative) else np.nan,
                      'share_agreeing_with_mouse_sign': float((np.sign(informative) == np.sign(mouse_mean.get(g, np.nan))).mean()) if len(informative) else np.nan})
c_post = pd.DataFrame(post_rows)
c_post.to_csv(tables / 'C_posthoc_centred_lead_genes.csv', index=False)
centred.loc[[g for g in LEADS_C + list(positive) if g in centred.index]].to_csv(tables / 'C_posthoc_centred_by_sample.csv')
display(c_post.round(2))

## D · Rat segment proteomics (Kidney Tubule Expression Atlas)

Copies per cell in microdissected rat S1, S2 and S3 (three samples each). Log ratios are centred on
the median ratio of proteins quantified in all nine PT samples, because PT segments differ in
overall protein yield.

In [ ]:
ktea = pd.read_csv(external / 'ktea_rat_proteomics' / 'KTEA_proteome_individual_sample.txt', sep='\t').drop_duplicates('gene').set_index('gene')
pt_cols = {s: [c for c in ktea.columns if c.startswith(f'{s}_')] for s in ('S1', 'S2', 'S3')}
block = ktea[sum(pt_cols.values(), [])].astype(float)
full = block[(block > 0).all(axis=1)]
centre = {(a, b): float(np.median(np.log2(full[pt_cols[a]].mean(axis=1)) - np.log2(full[pt_cols[b]].mean(axis=1))))
          for a, b in (('S3', 'S1'), ('S3', 'S2'), ('S2', 'S1'))}
D_GENES = ['Gatm', 'Gamt', 'Gclc', 'Gclm', 'Gss', 'Dcxr', 'Ugt3a1', 'Slc22a6', 'Slc22a7', 'Slc22a8', 'Acadm', 'Acaa2',
           'Rbp4', 'Cyp2e1', 'Slc6a19', 'Slc5a2', 'Slc5a1', 'Aqp1']
d_rows = []
for g in D_GENES:
    if g not in ktea.index:
        d_rows.append({'gene': g, 'note': 'not quantified'})
        continue
    m = {s: block.loc[g, pt_cols[s]].mean() for s in pt_cols}
    det = '/'.join(str(int((block.loc[g, pt_cols[s]] > 0).sum())) for s in pt_cols)
    row = {'gene': g, **{f'{s} copies/cell': m[s] for s in m}, 'detected S1/S2/S3 (of 3)': det}
    for a, b in centre:
        row[f'log2 {a}/{b} (centred)'] = (np.log2((m[a] + 1) / (m[b] + 1)) - centre[(a, b)])
    d_rows.append(row)
d_table = pd.DataFrame(d_rows)
d_table.to_csv(tables / 'D_rat_segment_proteome.csv', index=False)
print('Centring (median log2 ratios of fully quantified proteins):', {f'{a}/{b}': round(v, 2) for (a, b), v in centre.items()})
display(d_table.round(2))

## Figure · Revision checks

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 8.5, 'axes.titlesize': 9,
                     'axes.titleweight': 'bold', 'pdf.fonttype': 42})
HUMAN, MOUSE, INK, MUTED = '#D55E00', '#0072B2', '#333333', '#999999'
fig, axes = plt.subplots(1, 4, figsize=(17, 4.2), layout='constrained')
ax = axes[0]
for i, (program, row) in enumerate(state_programs.iterrows()):
    ax.scatter(row.median_abs_interaction, row.median_largest_state_effect, color=INK, s=28)
    ax.annotate(program, (row.median_abs_interaction, row.median_largest_state_effect), fontsize=7, xytext=(4, 2), textcoords='offset points')
lim = max(state_programs.median_abs_interaction.max(), state_programs.median_largest_state_effect.max()) * 1.15
ax.plot([0, lim], [0, .5 * lim], color=MUTED, ls='--', lw=.8)
ax.set(xlim=(0, lim), ylim=(0, lim), xlabel='Median |species × position interaction| (log2)',
       ylabel='Median largest fasting / time-of-day effect (log2)')
ax.set_title('a · Whole-kidney state effects', loc='left')
ax = axes[1]
zt4, zt16 = sn_contrast[['ZT4_1', 'ZT4_2']].mean(axis=1), sn_contrast[['ZT16_1', 'ZT16_2']].mean(axis=1)
tested = [g for g in sorted(TESTED) if g in sn_contrast.index]
ax.scatter(zt4[tested], zt16[tested], s=14, color=MOUSE)
for g in tested:
    if max(abs(zt4[g]), abs(zt16[g])) >= 1.5:
        ax.annotate(g, (zt4[g], zt16[g]), fontsize=6.5, xytext=(3, 2), textcoords='offset points')
lims = [min(zt4[tested].min(), zt16[tested].min()) - .3, max(zt4[tested].max(), zt16[tested].max()) + .3]
ax.plot(lims, lims, color=MUTED, lw=.8)
ax.axhline(0, color=MUTED, lw=.5)
ax.axvline(0, color=MUTED, lw=.5)
ax.set(xlabel='Mouse snRNA S3 − S1 at ZT4 (log2)', ylabel='S3 − S1 at ZT16 (log2)')
ax.set_title('b · Zonation at two times of day', loc='left')
ax = axes[2]
colors = {'normal': INK, 'chronic kidney disease': MUTED, 'acute kidney injury': HUMAN}
for disease, block_ in lake_scores.groupby('disease'):
    ax.scatter(block_.injury_score, block_.altered_fraction, s=20, color=colors.get(disease, MUTED), label=f'Lake {disease}')
for name, row in placement.iterrows():
    ax.axvline(row['calibrated score'], color=HUMAN, ls='--', lw=1)
ax.text(placement['calibrated score'].mean(), .98, ' our donor (both sections,\n mouse-calibrated)', va='top', ha='left',
        fontsize=7, color=HUMAN, transform=ax.get_xaxis_transform())
ax.set(xlabel='Injury score (aPT signature, rank-based)', ylabel='Altered PT fraction (Lake)')
ax.legend(frameon=False, fontsize=6.5, loc='lower right')
ax.set_title('c · Our donor on the Lake injury axis', loc='left')
ax = axes[3]
show = c_table.dropna(subset=['visium_mean']).set_index('gene')
y = np.arange(len(show))
for j, s in enumerate(st_samples):
    ax.scatter(centred.loc[show.index, s], y, s=10, color=INK if status[s] == 'Disease' else HUMAN, alpha=.6)
ax.scatter(show.lake_human_S3_minus_S1, y, marker='|', s=120, color=HUMAN, label='Lake (healthy)')
ax.scatter(show.mouse_sn_mean, y, marker='x', s=25, color=MOUSE, label='mouse snRNA')
ax.axvline(0, color=MUTED, lw=.6)
ax.set_yticks(y, show.index, fontsize=7)
ax.set(xlabel='S3 − S1 (log2); dots = Visium samples, centred (post hoc)', xlim=(-6, 6))
ax.legend(frameon=False, fontsize=6.5, loc='lower right')
ax.set_title('d · Visium donors (label check failed; post hoc)', loc='left')
for suffix in ('pdf', 'png'):
    fig.savefig(figures / f'fig_revision_state.{suffix}', bbox_inches='tight', dpi=200)
plt.show()
plt.close(fig)

In [ ]:
manifest = {'notebook_logic_version': NOTEBOOK_LOGIC_VERSION, 'S1_markers': S1_MARKERS, 'S3_markers': S3_MARKERS,
            'sn_nuclei': sn_counts, 'A1': state_programs.round(3).to_dict(orient='index'), 'A2': a2.round(3).astype(str).to_dict(orient='index'),
            'B': b_summary.astype(str).to_dict(), 'B_placement': placement.round(3).astype(str).to_dict(orient='index'),
            'C_samples': {s: status[s] for s in st_samples}, 'C_k_median': k_bar, 'C_label_check': label_check.mean(axis=1).round(3).to_dict(),
            'C_posthoc_label_check': label_check_c.mean(axis=1).round(3).to_dict(),
            'D_centring': {f'{a}/{b}': v for (a, b), v in centre.items()}}
(output / 'run_manifest.json').write_text(json.dumps(manifest, indent=2, default=str))
print('Wrote', sorted(p.name for p in tables.iterdir()))